# Step 2: H200×8 多卡部署 — 纯 TP 命令构造 + TP 选型判断

**目标**：掌握单节点 8 卡的纯张量并行（TP）部署——构造 `vllm serve` 命令模板，判断"什么时候该升 TP"（不是为了跑起来，而是为了放不下/提吞吐），识别 continuous batching/PagedAttention 默认开启，掌握 TP>1 的 NCCL 启动坑排查。

**对应 OUTLINE 课时**：4.2 H200×8 多卡 TP 部署（~50 分钟）。

> **声明式主线**：部署无需改模型代码——本节你写的只有 `vllm serve` 命令 + 客户端调用两类。L3 真起服务跨模块读 M2 7B 量化产物；0.5B 兜底命令构造验证。


## 学完应能讲清（学完本节应能口头回答）

1. 单节点 8 卡为什么用**纯 TP**（`--tensor-parallel-size`）不混 PP？PP 什么时候才有意义（跨节点）？（提示：PP 把层切到不同卡、流水线气泡浪费算力；单节点 TP 通信够快，纯 TP 更简单高效；PP 仅在跨节点带宽受限、单节点放不下时才有意义）
2. continuous batching / PagedAttention 要不要手动开？（V1 引擎默认开启，无需 flag——不要画蛇添足传 `--enable-chunked-prefill` 之类以为不开）
3. TP>1 启动 hang / rank timeout（NCCL）怎么排查？`--disable-custom-all-reduce` 什么时候用？`NCCL_DEBUG=INFO` / `VLLM_WORKER_MULTIPROC_METHOD=spawn` 各解什么？（提示：NCCL 通信死锁/版本不匹配、自定义 all-reduce 在某些拓扑不兼容、spawn 解多卡子进程 fork 问题）
4. Qwen2.5-7B 量化后**单卡就能放**（~7-14GB），TP=8 更多是教学演示——真实提吞吐的 TP=2 + 数据并行（多实例 + LB）思路是什么？（提示：单实例 TP 提升有上限，多实例各自小 TP + 前端负载均衡吞吐更高）


In [ ]:
%%capture
import pathlib, os, math, shlex
import ipytest
ipytest.autoconfig()


In [ ]:
# Setup cell（cwd 无关路径解析）。M4 跨模块读 M2/M3 7B 量化产物做 L3；0.5B 兜底 L2。
import pathlib

def _find_module_root(start):
    p = pathlib.Path(start).resolve()
    for cand in [p, *p.parents]:
        if (cand / "scripts").is_dir() and (cand / "steps").is_dir():
            return cand
    raise RuntimeError("找不到模块根（含 scripts/ + steps/ 的目录）")

MODULE_ROOT   = _find_module_root(pathlib.Path.cwd())
MODEL_DIR      = MODULE_ROOT / "models" / "Qwen2.5-7B-Instruct"          # FP16 基线
TINY_MODEL_DIR = MODULE_ROOT / "models" / "Qwen2.5-0.5B-Instruct"         # L2/L3 轻量验证兜底
OUT_ROOT       = MODULE_ROOT / "out"; OUT_ROOT.mkdir(parents=True, exist_ok=True)
REPO_COURSE = MODULE_ROOT.parent
M2_OUT = REPO_COURSE / "m2-quant-pipeline" / "out"      # qwen7b-fp8 / qwen7b-awq / qwen7b-smoothquant
M3_OUT = REPO_COURSE / "m3-tuning-eval" / "out"
print("MODULE_ROOT =", MODULE_ROOT)
print("M2_OUT =", M2_OUT, "| exists:", M2_OUT.exists())


## 原理：单节点纯 TP，不混 PP

**为什么单节点用纯 TP**：张量并行（TP）把每层权重切到多卡、各卡算一部分后 all-reduce 汇总——通信是层内集合通信（NCCL），单节点 NVLink/PCIe 带宽足够，气泡小。流水线并行（PP）把不同层切到不同卡、像流水线一样传递中间激活——层间依赖产生**气泡**（等前一阶段算完），单节点内 PP 的气泡浪费远大于 TP 的通信开销。**结论：单节点（尤其 8 卡）纯 TP 最优；PP 仅在跨节点（多机、节点间带宽受限、单机放不下）才有意义。**

**continuous batching / PagedAttention 默认开**：vLLM V1 引擎默认启用 continuous batching（请求级动态拼批）+ PagedAttention（分块 KV-Cache）。**无需任何 flag**——常见的画蛇添足是误传 `--enable-chunked-prefill`（其实默认就开）。教学重点是"别多此一举"。

**TP 通信的关键 flag**：
- `--disable-custom-all-reduce`：禁用 vLLM 自带的 custom all-reduce kernel，回退到 NCCL。**何时用**：当自定义 all-reduce 在你的 GPU 拓扑（如无 NVLink 的 PCIe-only 节点）上不兼容、启动 hang 或结果错误时。
- `--enable-prefix-caching`：缓存重复前缀（系统 prompt 等）的 KV，提吞吐。默认 V1 已开，显式传无害。
- 环境变量排查 TP 启动坑（见 s4 详解）：
  - `NCCL_DEBUG=INFO`：打印 NCCL 通信细节，定位 rank timeout / 死锁根因。
  - `VLLM_WORKER_MULTIPROC_METHOD=spawn`：多卡 worker 子进程用 spawn（非 fork），解决 fork 后 CUDA 上下文 / NCCL 状态损坏导致的 hang。
  - `pip show nvidia-nccl-cu12`：查 NCCL 版本是否与 vLLM wheel 配对的 CUDA 一致。

**TP 不是越大越好**：Qwen2.5-7B 量化后权重仅 ~7-14GB，单张 H200（141GB）轻松放下——TP=1 就能跑。TP=8（教学演示）反而因 all-reduce 通信开销未必更快。**真实提吞吐**的姿势：单实例 TP=2（够放 + 一点并行）× 多实例（数据并行）+ 前端 LB（nginx/HAProxy 轮询），比单实例 TP=8 吞吐更高、容错更好。


## 亲手摸一摸：GPU 拓扑 + vllm serve 参数

看本机 GPU 数与 P2P 拓扑、vllm serve 支持哪些并行 flag——为构造命令打底。


In [ ]:
# 摸一摸：nvidia-smi 拓扑 + vllm serve --help parallel 相关（不真起服务）
import subprocess
# nvidia-smi 拓扑（GPU 数 + P2P）
try:
    out = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True, timeout=10)
    gpus = [l for l in out.stdout.splitlines() if l.startswith("GPU")]
    print("本机 GPU 数:", len(gpus))
    for g in gpus[:8]:
        print(" ", g)
except Exception as e:
    print("nvidia-smi 不可用（CPU 环境/无 GPU）:", e)
    print("（H200×8 节点会有 8 行 GPU i；本步逻辑层不依赖真 GPU）")

# vllm serve --help 看支持的并行/缓存 flag（只 grep，不真起）
print("\n=== vllm serve 并行/缓存相关 flag（从 --help grep）===")
help_flags = [
    "--tensor-parallel-size", "--pipeline-parallel-size",
    "--enable-prefix-caching", "--disable-custom-all-reduce",
    "--gpu-memory-utilization", "--max-model-len",
]
for f in help_flags:
    print(" ", f)
print("（continuous batching / PagedAttention V1 默认开启，无对应 flag）")


## 本步填空（2 个）

1. **`build_serve_cmd(model_path, tp, gpu_mem_util=0.90, max_model_len=32768, extra=())`** — 构造完整 `vllm serve` 命令字符串（含 `--tensor-parallel-size`、`--gpu-memory-utilization`、`--max-model-len`、`--enable-prefix-caching`、`--disable-custom-all-reduce`）。**为什么这么设计（填前先想）**：4.2 的核心交付是这条命令模板；学员亲手组装 flag 理解每个参数作用，而不是抄。`extra` 透传额外 flag（如 `--dtype auto`）。
2. **`recommend_tp(num_gpus, model_weight_gb, gpu_mem_gb, headroom=0.0)`**（判断型）— 判断推荐 TP 数（量化 7B ~7-14GB 单卡能放 → TP 可小（1）；放不下 → 升 TP；不超过 `num_gpus`）。**为什么这么设计**：判断型——让学员想清"TP 不是越大越好，是为了放不下或提吞吐才用"。


In [ ]:
def build_serve_cmd(model_path, tp, gpu_mem_util=0.90, max_model_len=32768, extra=()):
    """构造完整 vllm serve 命令字符串（空格拼接，model_path 等含空格的用 shlex.quote）。

    为什么这么设计（填前先想）：
    - 这是 4.2 的核心交付物——一条可复现的 serve 命令。每个 flag 对应一个工程决策：
        --tensor-parallel-size <tp>          TP 数（单节点纯 TP，不混 PP）
        --gpu-memory-utilization <f>         vLLM 可用显存占比（留余量给系统，0.90 常用）
        --max-model-len <n>                  最大上下文（KV-Cache 上限，影响 OOM 风险）
        --enable-prefix-caching              前缀缓存（V1 默认开，显式传无害）
        --disable-custom-all-reduce          禁用自定义 all-reduce 回退 NCCL（拓扑兼容兜底）
    - continuous batching / PagedAttention 默认开，不要画蛇添足传对应 flag。
    - extra：透传额外 flag（list/tuple），如 ['--dtype','auto']。

    返回：命令字符串（如 'vllm serve /path --tensor-parallel-size 2 ...'）。
    """
    # TODO: 用 shlex 组装。parts 从 ['vllm','serve',str(model_path)] 起，
    #   追加上面 5 组 flag（flag 名 + 值成对）。extra 追加到末尾。
    #   最后 ' '.join——值里有空格/等号的用 shlex.quote，其余裸拼。
    raise NotImplementedError


In [ ]:
%%ipytest -qq
# L1 测试（build_serve_cmd）——填完 build_serve_cmd 立即单独跑此 cell 验证（不依赖 recommend_tp）。

def test_build_serve_cmd_has_flags():
    cmd = build_serve_cmd("/models/qwen7b-smoothquant", tp=2)
    assert cmd.startswith("vllm serve")
    assert "/models/qwen7b-smoothquant" in cmd
    assert "--tensor-parallel-size 2" in cmd
    assert "--gpu-memory-utilization 0.9" in cmd       # 默认 0.90 -> "0.9"
    assert "--max-model-len 32768" in cmd
    assert "--enable-prefix-caching" in cmd
    assert "--disable-custom-all-reduce" in cmd

def test_build_serve_cmd_custom():
    cmd = build_serve_cmd("/m", tp=1, gpu_mem_util=0.85, max_model_len=8192)
    assert "--tensor-parallel-size 1" in cmd
    assert "--gpu-memory-utilization 0.85" in cmd
    assert "--max-model-len 8192" in cmd

def test_build_serve_cmd_extra():
    cmd = build_serve_cmd("/m", tp=8, extra=["--dtype", "auto"])
    assert "--dtype auto" in cmd
    assert "--tensor-parallel-size 8" in cmd

def test_build_serve_cmd_path_quoted_if_space():
    cmd = build_serve_cmd("/path with space/model", tp=1)
    assert "/path with space/model" in cmd  # 原始路径仍在

In [ ]:
def recommend_tp(num_gpus, model_weight_gb, gpu_mem_gb, headroom=0.0):
    """判断推荐 TP 数（判断型）。

    为什么这么设计（填前先想）：
    - TP 不是越大越好——它增加 all-reduce 通信开销。升 TP 的唯一理由是「单卡放不下」或「要提吞吐」。
    - 单卡能放下模型权重（model_weight_gb <= gpu_mem_gb * (0.9 + headroom)）→ TP=1 最省（教学可手动升）。
      0.9 是 vLLM gpu_memory_utilification 默认占比（留 10% 给系统）；headroom 允许略松。
    - 放不下 → TP = ceil(model_weight_gb / (gpu_mem_gb * 0.9))，但不超过 num_gpus（单节点 TP 上限）。
    - 注意：这只算权重；KV-Cache 还要占显存，故实战中 TP 常比纯权重算出的大一档。
    - PP（流水线并行）仅在跨节点才有意义——本函数只返回单节点 TP。

    返回：int（1 <= tp <= num_gpus）。
    """
    # TODO:
    #   1) per_gpu_budget = gpu_mem_gb * (0.9 + headroom)
    #   2) needed = ceil(model_weight_gb / per_gpu_budget)（budget>0 时；否则 num_gpus）
    #   3) needed <= 1 → return 1
    #   4) tp = min(needed, num_gpus) → return tp
    raise NotImplementedError


In [ ]:
%%ipytest -qq
# L1 测试（recommend_tp）——填完 recommend_tp 立即单独跑此 cell 验证（不依赖 build_serve_cmd）。

def test_recommend_tp_single_card_fits():
    # 7B SmoothQuant W8A8 ~7.5GB，H200 141GB 单卡轻松放下 -> TP=1
    assert recommend_tp(num_gpus=8, model_weight_gb=8, gpu_mem_gb=141) == 1

def test_recommend_tp_does_not_fit():
    # 70B FP16 ~140GB，141GB 卡按 0.9 占比 = 126.9GB 放不下 -> ceil(140/126.9)=2
    assert recommend_tp(num_gpus=8, model_weight_gb=140, gpu_mem_gb=141) == 2

def test_recommend_tp_capped_by_num_gpus():
    # 需要 4 但只有 2 卡 -> 返回 2（提示需更多卡或 PP）
    assert recommend_tp(num_gpus=2, model_weight_gb=400, gpu_mem_gb=141) == 2

def test_recommend_tp_minimum_one():
    # 极小模型
    assert recommend_tp(num_gpus=8, model_weight_gb=1, gpu_mem_gb=141) == 1

## L2（CPU）：命令构造 + TP 选型判断（纯逻辑）

L2 验 `build_serve_cmd` 命令模板正确、`recommend_tp` 判断边界合理（CPU 可跑，不真起服务）。配合真实 7B 产物路径演示命令落地。


In [ ]:
## L2：构造 SmoothQuant serve 命令 + TP 选型判断（跨模块 M2/M3 产物路径）
# 本模块只部署 SmoothQuant W8A8：M2 全量化 + M3 调优后 final（缺产物用 FP16 基线兜底演示命令构造）
candidates = {
    "M2 SmoothQuant": M2_OUT / "qwen7b-smoothquant",
    "M3 final (调优后)": M3_OUT / "s6_final",
}
demo = {name: (d if (d / "config.json").exists() else MODEL_DIR) for name, d in candidates.items()}

print("=== L2：SmoothQuant serve 命令（TP=2，教学演示）===")
for name, path in demo.items():
    cmd = build_serve_cmd(str(path), tp=2)
    is_real = candidates[name] != MODEL_DIR and (candidates[name] / "config.json").exists()
    tag = "（真 W8A8 产物）" if is_real else "（FP16 兜底，先跑 M2 s5/M3 s6）"
    print("\n[%s] %s" % (name, tag))
    print("  " + cmd)

print("\n=== L2：TP 选型判断 ===")
cases = [
    ("7B SmoothQuant W8A8 (~8GB，单卡放得下)", dict(num_gpus=8, model_weight_gb=8, gpu_mem_gb=141)),
    ("70B FP16 (~140GB，单卡放不下)",          dict(num_gpus=8, model_weight_gb=140, gpu_mem_gb=141)),
]
for label, kw in cases:
    tp = recommend_tp(**kw)
    print("  %s -> TP=%d" % (label, tp))
assert recommend_tp(num_gpus=8, model_weight_gb=8, gpu_mem_gb=141) == 1, "7B W8A8 单卡放得下应 TP=1"
print("\nL2 通过：serve 命令模板含全部关键 flag、TP 判断逻辑正确（真起服务见 L3）。")

## L3（H200，GPU + SKIP_L3 双守卫）：真 vllm serve + 客户端调用

L3 真起 vllm 服务（TP=2 或教学 TP=8）+ 用 OpenAI 兼容客户端发一句，验证多卡部署闭环。

> **不在 notebook 里后台 `vllm serve &`**（避免后台进程残留/端口占用/kill 失败）。L3 用 `subprocess` 同步起服务、测完即 kill；命令由 `build_serve_cmd` 构造。
>
> **L3 双守卫**：`torch.cuda.is_available() and not os.environ.get('SKIP_L3')`——reviewer 执行验证设 `SKIP_L3=1` 跳过（真起服务 + 多卡分钟级，太重）；真人跑时不设，L3 实证。


In [ ]:
import torch, os, subprocess, time, signal

def run_l3_multigpu_serve():
    import torch
    ngpu = torch.cuda.device_count()
    if ngpu < 2:
        print("[L3] 本机 GPU<2，TP=2 不可行；改用 0.5B 单卡 serve 演示 serve+客户端闭环。")
        model = str(TINY_MODEL_DIR); tp = 1
    else:
        # 本模块只部署 SmoothQuant W8A8：优先 M3 final（调优后），缺则 M2 全量化，再缺 0.5B 兜底
        for d in [M3_OUT / "s6_final", M2_OUT / "qwen7b-smoothquant"]:
            if (d / "config.json").exists():
                model = str(d); break
        else:
            model = str(TINY_MODEL_DIR)
            print("[L3] M2/M3 SmoothQuant W8A8 产物均缺失，用 0.5B 兜底（TP 演示逻辑不变）。")
        tp = min(2, ngpu)
    cmd = build_serve_cmd(model, tp=tp)
    print("[L3] 起服务：", cmd)
    # 同步起服务（前台子进程），轮询 /health 就绪后发请求，测完 kill
    proc = subprocess.Popen(cmd, shell=True, preexec_fn=os.setsid,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    try:
        import urllib.request, json as _json
        url = "http://localhost:8000"
        ready = False
        for _ in range(120):  # 最多等 120s
            try:
                r = urllib.request.urlopen(url + "/health", timeout=2)
                if r.status == 200:
                    ready = True; break
            except Exception:
                time.sleep(2)
        if not ready:
            print("[L3] 服务 120s 未就绪（多卡 NCCL 初始化慢属正常）——见日志排查 TP 启动坑（s4）。")
            return
        # OpenAI 兼容客户端发一句
        req = urllib.request.Request(url + "/v1/completions",
            data=_json.dumps({"model": model, "prompt": "你好", "max_tokens": 16, "temperature": 0}).encode(),
            headers={"Content-Type": "application/json"})
        resp = _json.loads(urllib.request.urlopen(req, timeout=60).read())
        print("[L3] 客户端输出:", resp["choices"][0]["text"].strip())
        print("[L3] serve+客户端闭环成功（TP=%d，SmoothQuant W8A8 走 auto 无需 --quantization flag）。" % tp)
    finally:
        os.killpg(os.getpgid(proc.pid), signal.SIGTERM)

if torch.cuda.is_available() and not os.environ.get('SKIP_L3'):
    run_l3_multigpu_serve()
else:
    print("跳过 L3：无 GPU 或 SKIP_L3=1（reviewer 执行验证跳过真 vllm serve；真人跑时不设 SKIP_L3，L3 实证）。")

## 产物检查：TP 启动成功 = 多卡部署闭环

L3 跑通即证明：单节点纯 TP（无 PP）+ vLLM 默认 continuous batching/PagedAttention 就能让多卡协同服务请求——**无需改模型代码、无需手写 all-reduce**（vLLM + NCCL 全包）。

**回顾 TP 坑排查清单**（若 L3 启动 hang，按此查）：
1. `NCCL_DEBUG=INFO` 看通信细节 → rank timeout 多半是版本不匹配或拓扑不兼容。
2. `--disable-custom-all-reduce` 回退 NCCL（自定义 all-reduce 在某些 PCIe-only 拓扑不兼容）。
3. `VLLM_WORKER_MULTIPROC_METHOD=spawn` 解决 fork 后 CUDA 上下文损坏的 hang。
4. `pip show nvidia-nccl-cu12` 查 NCCL 版本与 vLLM wheel 配对的 CUDA 是否一致。
5. TP>1 慢启动（几十秒 NCCL 初始化）属正常——耐心等 `/health`，别误判为 hang kill 掉。

下一步 s3 在这套部署上做性能压测。
